<a href="https://colab.research.google.com/github/descruceria777/Se-alesySistemas/blob/main/proyecto_sys_2025_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#instalación de librerías
!pip install streamlit -q


In [ ]:
import os
os.makedirs('pages', exist_ok=True)

In [ ]:
import os
os.makedirs('pages', exist_ok=True)

## Crear archivos de página para cada fase

### Subtask:
Crear archivos Python separados en la carpeta `pages` para cada una de las cuatro fases de simulación (`1_Dominio_Frecuencia.py`, `2_Senales_IQ.py`, `3_Modulacion_QAM.py`, `4_Sistema_Completo_QAM.py`).


**Reasoning**:
Create the four Python files in the 'pages' directory for each simulation phase.



In [ ]:
%%writefile pages/1_Dominio_Frecuencia.py
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal import butter, lfilter, freqz

st.set_page_config(
    page_title="Dominio de la Frecuencia",
    page_icon="📊",
    layout="wide"
)

st.markdown("# 1. El Dominio de la Frecuencia (FFT y Filtrado) 📊")
st.write(
    """
    Aquí puedes observar cómo la Transformada Rápida de Fourier (FFT) descompone una señal en sus componentes de frecuencia.
    También se demuestra el efecto de un filtro paso-bajo en la señal, permitiendo seleccionar o eliminar ciertas frecuencias.
    """
)

st.markdown("### Conceptos Clave en esta Sección 💡") # Añadir emoji
st.markdown("""
- **Dominio del Tiempo vs. Dominio de la Frecuencia:** Las señales pueden representarse como cambian con el tiempo (dominio del tiempo) o como una combinación de frecuencias (dominio de la frecuencia). 🕰️↔️🎶
- **Transformada de Fourier (FFT):** Es una herramienta matemática que nos permite pasar de la representación en el dominio del tiempo a la representación en el dominio de la frecuencia. Nos muestra qué frecuencias están presentes en una señal y con qué amplitud. ➡️📈📊
- **Filtrado Digital:** Proceso para modificar el contenido en frecuencia de una señal, atenuando o realzando ciertas frecuencias. Un filtro paso-bajo permite pasar frecuencias por debajo de una frecuencia de corte y atenúa las frecuencias por encima. 🧹🎶
- **Diagrama de Bode:** Representación gráfica de la respuesta en frecuencia de un sistema (como un filtro). Generalmente consta de dos gráficos: uno de la magnitud (ganancia) en decibeles (dB) vs. frecuencia, y otro de la fase en grados o radianes vs. frecuencia. 📉📐
""")


# Parámetros para la simulación de frecuencia
st.markdown("### Parámetros de la Señal y el Filtro ⚙️") # Añadir emoji
fs_freq = st.slider("Frecuencia de muestreo (Hz) ⏱️", 1000, 10000, 1000, key='fs_freq_slider_tab1') # Añadir emoji
t_freq = np.linspace(0, 1, fs_freq, endpoint=False)

# Señal sintética: suma de dos senoides
f1 = st.slider("Frecuencia de la primera sinusoide (Hz) 🎵", 1, fs_freq//2 - 1, 5, key='freq1_slider_tab1') # Añadir emoji
f2 = st.slider("Frecuencia de la segunda sinusoide (Hz) 🎶", 1, fs_freq//2 - 1, 50, key='freq2_slider_tab1') # Añadir emoji
amplitude1 = st.slider("Amplitud de la primera sinusoide 💪", 0.1, 2.0, 1.0, key='amp1_slider_tab1') # Añadir emoji
amplitude2 = st.slider("Amplitud de la segunda sinusoide ✨", 0.1, 2.0, 0.5, key='amp2_slider_tab1') # Añadir emoji

signal_freq = amplitude1 * np.sin(2 * np.pi * f1 * t_freq) + amplitude2 * np.sin(2 * np.pi * f2 * t_freq)

# Aplicar FFT
yf_freq = fft(signal_freq)
xf_freq = fftfreq(fs_freq, 1/fs_freq)[:fs_freq//2] # Solo mostrar frecuencias positivas

# Diseño y aplicación de filtro paso-bajo
cutoff_freq = st.slider("Frecuencia de corte del filtro paso-bajo (Hz) ✂️", 1, fs_freq//2 - 1, min(f1, f2) + 5, key='cutoff_slider_tab1') # Añadir emoji
order = st.slider("Orden del filtro 🔢", 1, 10, 5, key='order_slider_tab1') # Añadir emoji

nyquist = 0.5 * fs_freq
normal_cutoff = cutoff_freq / nyquist

b, a = butter(order, normal_cutoff, btype='low', analog=False)

filtered_signal_freq = lfilter(b, a, signal_freq)

# Graficar señal original y filtrada en el tiempo
st.subheader("Visualización en el Dominio del Tiempo 📊") # Añadir emoji
fig_time_freq, ax_time_freq = plt.subplots()
ax_time_freq.plot(t_freq, signal_freq, label='Señal Original')
ax_time_freq.plot(t_freq, filtered_signal_freq, label='Señal Filtrada')
ax_time_freq.set_xlabel("Tiempo [s]")
ax_time_freq.set_ylabel("Amplitud")
ax_time_freq.set_title("Señal Original y Filtrada en el Tiempo")
ax_time_freq.legend()
ax_time_freq.grid(True)
st.pyplot(fig_time_freq)

# Graficar espectro de frecuencia (FFT)
st.subheader("Visualización en el Dominio de la Frecuencia 📈") # Añadir emoji
st.write("El espectro muestra la 'cantidad' de cada frecuencia presente en la señal.")
fig_freq_spec, (ax_orig, ax_filt) = plt.subplots(2, 1, sharex=True, figsize=(10, 8))

# Espectro de la señal original
ax_orig.plot(xf_freq, 2.0/fs_freq * np.abs(yf_freq[0:fs_freq//2]))
ax_orig.set_ylabel("Amplitud Normalizada")
ax_orig.set_title("Espectro de Frecuencia de la Señal Original")
ax_orig.grid()

# Espectro de la señal filtrada
yf_filt_freq = fft(filtered_signal_freq)
ax_filt.plot(xf_freq, 2.0/fs_freq * np.abs(yf_filt_freq[0:fs_freq//2]), color='orange')
ax_filt.set_xlabel("Frecuencia [Hz]")
ax_filt.set_ylabel("Amplitud Normalizada")
ax_filt.set_title("Espectro de Frecuencia de la Señal Filtrada")
ax_filt.grid()

st.pyplot(fig_freq_spec)


# Graficar respuesta en frecuencia del filtro (Diagrama de Bode - Amplitud y Fase)
st.subheader("Respuesta en Frecuencia del Filtro Paso-Bajo (Diagrama de Bode) 📉📐") # Añadir emoji
st.write("El Diagrama de Bode muestra cómo el filtro afecta la magnitud y la fase de las diferentes frecuencias.")
w, h = freqz(b, a, worN=8000, fs=fs_freq)
frequencies = w

fig_bode, (ax_mag, ax_phase) = plt.subplots(2, 1, sharex=True, figsize=(10, 8))

# Diagrama de Bode - Amplitud
ax_mag.plot(frequencies, 20 * np.log10(abs(h)))
ax_mag.set_ylabel("Ganancia [dB]")
ax_mag.set_title("Diagrama de Bode (Amplitud) del Filtro Paso-Bajo")
ax_mag.grid(True, which='both', linestyle='-', linewidth=0.5)
ax_mag.axvline(cutoff_freq, color='red', linestyle='--', label=f'Frecuencia de Corte ({cutoff_freq} Hz)')
ax_mag.legend()

# Diagrama de Bode - Fase
angles = np.unwrap(np.angle(h)) # Desenvuelve la fase para evitar saltos de 2*pi
ax_phase.plot(frequencies, angles * 180 / np.pi, color='green') # Convertir a grados
ax_phase.set_xlabel("Frecuencia [Hz]")
ax_phase.set_ylabel("Fase [grados]")
ax_phase.set_title("Diagrama de Bode (Fase) del Filtro Paso-Bajo")
ax_phase.grid(True, which='both', linestyle='-', linewidth=0.5)
ax_phase.axvline(cutoff_freq, color='red', linestyle='--', label=f'Frecuencia de Corte ({cutoff_freq} Hz)')


st.pyplot(fig_bode)

Overwriting pages/1_Dominio_Frecuencia.py


**Reasoning**:
Create the second Python file in the 'pages' directory for the I/Q signals simulation.



In [ ]:
%%writefile pages/2_Senales_IQ.py
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal import hilbert, freqz # Importar freqz para mostrar respuesta en frecuencia si aplica

st.set_page_config(
    page_title="Señales I/Q",
    page_icon="📈",
    layout="wide"
)

st.markdown("# 2. Construyendo las Señales I/Q 📈")
st.write(
    """
    Aquí se muestra cómo obtener la componente en Cuadratura (Q) de una señal a partir de su componente En-fase (I)
    utilizando la Transformada de Hilbert. La señal analítica resultante tiene la forma I(t) + jQ(t).
    Estas componentes son fundamentales para la modulación en cuadratura como QAM.
    """
)

st.markdown("### Conceptos Clave en esta Sección 💡")
st.markdown("""
- **Señal Analítica:** Una señal compleja donde la parte real es la señal original (componente En-fase, I) y la parte imaginaria es su Transformada de Hilbert (componente en Cuadratura, Q). Es muy útil en comunicaciones para representar señales de banda paso. 🧠➕📏
- **Transformada de Hilbert:** Es una operación matemática que desfasa cada componente de frecuencia de una señal real en -90 grados (-π/2 radianes) para frecuencias positivas y +90 grados (+π/2 radianes) para frecuencias negativas. Nos ayuda a obtener la componente Q a partir de la componente I. 📐🔄
- **Componentes En-fase (I) y en Cuadratura (Q):** Son dos señales reales que, cuando se combinan (I + jQ), forman la señal analítica. Representan la amplitud de la señal compleja en dos ejes ortogonales (como ejes X e Y en un plano complejo). Son esenciales para la modulación en cuadratura. ↔️↕️
""")


# Parámetros para la simulación I/Q
st.markdown("### Parámetros de la Señal Mensaje ⚙️")
fs_iq = st.slider("Frecuencia de muestreo (Hz) ⏱️", 1000, 10000, 1000, key='fs_iq_slider_tab2')
t_iq = np.linspace(0, 1, fs_iq, endpoint=False)

# Señal mensaje (componente I)
message_freq = st.slider("Frecuencia de la señal mensaje (Hz) 🎵", 1, fs_iq//2 -1, 10, key='message_freq_slider_tab2')
message_amplitude = st.slider("Amplitud de la señal mensaje 💪", 0.1, 2.0, 1.0, key='message_amp_slider_tab2')

signal_i = message_amplitude * np.sin(2 * np.pi * message_freq * t_iq)

# Aplicar Transformada de Hilbert para obtener la señal Q
analytic_signal_iq = hilbert(signal_i)
signal_q = analytic_signal_iq.imag


# Graficar Señales I y Q en el tiempo
st.subheader("Señales En-fase (I) y Cuadratura (Q) en el Tiempo 📊")
fig_iq_time, ax_iq_time = plt.subplots()
ax_iq_time.plot(t_iq, signal_i, label='Señal En-fase (I)')
ax_iq_time.plot(t_iq, signal_q, label='Señal en Cuadratura (Q)')
ax_iq_time.set_xlabel("Tiempo [s]")
ax_iq_time.set_ylabel("Amplitud")
ax_iq_time.set_title("Señales I y Q Derivadas por Transformada de Hilbert")
ax_iq_time.legend()
ax_iq_time.grid(True)
st.pyplot(fig_iq_time)

# Graficar relación I vs Q (Visualización del Desfase)
st.subheader("Visualización del Desfase (Gráfica Q vs. I) 🔄")
st.write("Este gráfico muestra la relación entre las señales I y Q en el tiempo. Para una sinusoide simple, se espera ver una forma elíptica o circular, indicando el desfase de 90 grados.")
fig_iq_scatter, ax_iq_scatter = plt.subplots()
ax_iq_scatter.scatter(signal_i, signal_q, s=5, alpha=0.5)
ax_iq_scatter.set_xlabel("Componente En-fase (I)")
ax_iq_scatter.set_ylabel("Componente en Cuadratura (Q)")
ax_iq_scatter.set_title("Gráfica Q vs. I (Visualización del Desfase)")
ax_iq_scatter.grid(True)
ax_iq_scatter.set_aspect('equal', adjustable='box')
max_amp = max(np.max(np.abs(signal_i)), np.max(np.abs(signal_q))) * 1.1
ax_iq_scatter.set_xlim(-max_amp, max_amp)
ax_iq_scatter.set_ylim(-max_amp, max_amp)
st.pyplot(fig_iq_scatter)


# Graficar Espectro de Frecuencia de Señales I y Q
st.subheader("Espectro de Frecuencia de Señales I y Q 📈")
yf_i = fft(signal_i)
yf_q = fft(signal_q)
xf_iq = fftfreq(fs_iq, 1/fs_iq)[:fs_iq//2]

fig_iq_freq, ax_iq_freq = plt.subplots()
ax_iq_freq.plot(xf_iq, 2.0/fs_iq * np.abs(yf_i[0:fs_iq//2]), label='Espectro I')
ax_iq_freq.plot(xf_iq, 2.0/fs_iq * np.abs(yf_q[0:fs_iq//2]), label='Espectro Q')
ax_iq_freq.set_xlabel("Frecuencia [Hz]")
ax_iq_freq.set_ylabel("Amplitud Normalizada")
ax_iq_freq.set_title("Espectro de Frecuencia de Señales I y Q")
ax_iq_freq.legend()
ax_iq_freq.grid(True)
st.pyplot(fig_iq_freq)

# --- Visualización de la Respuesta en Frecuencia de la Transformada de Hilbert ---
st.subheader("Respuesta en Frecuencia de la Transformada de Hilbert")
st.write("La Transformada de Hilbert desfasa las componentes de frecuencia en ±90 grados.")

# Para visualizar la respuesta en frecuencia de la transformada de Hilbert
# en tiempo discreto, podemos usar la transformada de Fourier de la secuencia
# que representa la operación (por ejemplo, la respuesta al impulso de un filtro Hilbert ideal).
# Sin embargo, una forma más directa y conceptualmente simple es mostrar la
# magnitud (constante) y la fase (+90/-90).

# Creamos un vector de frecuencias
frequencies_hilbert = np.linspace(-fs_iq/2, fs_iq/2, 500) # Frecuencias positivas y negativas

# Magnitud ideal (constante, aquí mostramos 1)
magnitude_hilbert = np.ones_like(frequencies_hilbert)

# Fase ideal (+pi/2 para f > 0, -pi/2 para f < 0, 0 para f=0)
phase_hilbert = np.zeros_like(frequencies_hilbert)
phase_hilbert[frequencies_hilbert > 0] = np.pi/2 # +90 grados para frecuencias positivas
phase_hilbert[frequencies_hilbert < 0] = -np.pi/2 # -90 grados para frecuencias negativas

fig_hilbert_resp, (ax_mag_hilbert, ax_phase_hilbert) = plt.subplots(2, 1, sharex=True, figsize=(10, 8))

# Gráfico de Magnitud
ax_mag_hilbert.plot(frequencies_hilbert, magnitude_hilbert)
ax_mag_hilbert.set_ylabel("Magnitud")
ax_mag_hilbert.set_title("Respuesta en Frecuencia de la Transformada de Hilbert (Magnitud)")
ax_mag_hilbert.grid(True)
ax_mag_hilbert.set_ylim(-0.1, 1.1) # Ajustar límites para que se vea mejor la constante 1

# Gráfico de Fase
ax_phase_hilbert.plot(frequencies_hilbert, phase_hilbert * 180/np.pi, color='green') # Convertir a grados
ax_phase_hilbert.set_xlabel("Frecuencia [Hz]")
ax_phase_hilbert.set_ylabel("Fase [grados]")
ax_phase_hilbert.set_title("Respuesta en Frecuencia de la Transformada de Hilbert (Fase)")
ax_phase_hilbert.grid(True)
ax_phase_hilbert.set_yticks([-90, 0, 90]) # Marcar los valores clave de fase
ax_phase_hilbert.set_ylim(-100, 100) # Ajustar límites

st.pyplot(fig_hilbert_resp)

Overwriting pages/2_Senales_IQ.py


**Reasoning**:
Create the third Python file in the 'pages' directory for the QAM modulation simulation.



In [ ]:
%%writefile pages/3_Modulacion_QAM.py
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq

st.set_page_config(
    page_title="Modulación QAM",
    page_icon="🌌", # Emoji para QAM (constelación)
    layout="wide"
)

st.markdown("# 3. Modulación QAM 🌌") # Añadir emoji al título principal
st.write(
    """
    Esta sección demuestra el proceso de Modulación de Amplitud en Cuadratura (QAM).
    Se genera una secuencia de símbolos, se mapean a puntos de constelación,
    se construyen las señales en fase (I) y cuadratura (Q) correspondientes,
    y se modulan sobre una portadora para formar la señal QAM en el dominio del tiempo.
    También se visualiza su espectro.
    """
)

st.markdown("### Conceptos Clave en esta Sección ✨") # Añadir emoji
st.markdown("""
- **Modulación QAM (Quadrature Amplitude Modulation):** Técnica que transmite información variando tanto la amplitud como la fase de una señal portadora. Utiliza dos portadoras (una coseno y otra seno) desfasadas 90 grados. 📡🔄
- **Diagrama de Constelación:** Una representación gráfica de los posibles símbolos que pueden ser transmitidos. Cada punto en el diagrama representa un símbolo QAM único, definido por sus valores de amplitud en las componentes I (eje horizontal) y Q (eje vertical). ✨📍
- **Mapeo de Símbolos:** Proceso de asignar secuencias de bits a puntos específicos en el diagrama de constelación. 🗺️➡️📍
- **Señales Portadoras:** Señales de alta frecuencia (coseno y seno) que "transportan" la información modulada. La señal I(t) modula la portadora coseno y la señal Q(t) modula la portadora seno. 🌊🎶
""")

# Parámetros para la Modulación QAM (simulación básica en esta pestaña)
st.markdown("### Parámetros de Simulación y Portadora ⚙️") # Añadir emoji
fs_qam_mod = st.slider("Frecuencia de muestreo para Modulación (Hz) ⏱️", 5000, 50000, 10000, key='fs_qam_mod_slider_tab3') # Añadir emoji
duration_mod = st.slider("Duración de la señal (segundos) ⏳", 0.001, 0.05, 0.01, key='duration_mod_slider_tab3') # Añadir emoji - Un corto intervalo de tiempo para visualizar
t_qam_mod = np.linspace(0, duration_mod, int(fs_qam_mod * duration_mod), endpoint=False)

carrier_freq = st.slider("Frecuencia de la portadora (Hz) 📶", 100, 2000, 500, key='carrier_freq_slider_tab3') # Añadir emoji

# Ejemplo básico de símbolos QAM (simplificado para demostración)
# Usaremos un ejemplo de 16-QAM con unos pocos símbolos para visualizar la forma de onda.
qam_order_mod = 16 # Ejemplo fijo para esta sección
st.markdown(f"#### Diagrama de Constelación para {qam_order_mod}-QAM")
st.write(f"Este es el conjunto de puntos (símbolos) que pueden ser transmitidos con {qam_order_mod}-QAM.")
constellation_mod = np.array([
    -3-3j, -3-1j, -3+3j, -3+1j,
    -1-3j, -1-1j, -1+3j, -1+1j,
     3-3j, 3-1j, 3+3j, 3+1j,
    1-3j, 1-1j, 1+3j, 1+1j
]) * 1/np.sqrt(10) # Normalizado para energía promedio 1

# Visualizar la constelación ideal
fig_const_ideal, ax_const_ideal = plt.subplots(figsize=(6, 6))
ax_const_ideal.scatter(constellation_mod.real, constellation_mod.imag, s=100, edgecolors='black', marker='o')
for i, txt in enumerate(constellation_mod):
    ax_const_ideal.annotate(f'{i}', (txt.real, txt.imag), textcoords="offset points", xytext=(0,10), ha='center') # Mostrar índice del símbolo
ax_const_ideal.set_title(f"Constelación Ideal {qam_order_mod}-QAM")
ax_const_ideal.set_xlabel("En-fase (I)")
ax_const_ideal.set_ylabel("Cuadratura (Q)")
ax_const_ideal.grid(True)
ax_const_ideal.set_aspect('equal', adjustable='box')
max_val = np.max(np.abs(constellation_mod)) * 1.5
ax_const_ideal.set_xlim(-max_val, max_val)
ax_const_ideal.set_ylim(-max_val, max_val)
st.pyplot(fig_const_ideal)


st.markdown("#### Simulación con Símbolos de Ejemplo")
st.write("Selecciona unos pocos símbolos de la constelación para visualizar cómo se construyen las señales I(t) y Q(t) y la señal QAM modulada en el tiempo.")
# Secuencia de símbolos de ejemplo (usamos solo unos pocos símbolos para que la forma de onda sea visible)
# Puedes cambiar los índices para ver diferentes transiciones de símbolos
example_symbol_indices = st.multiselect("Selecciona índices de símbolos de ejemplo (hasta 4) 📍",
                                         list(range(len(constellation_mod))),
                                         default=[5, 14, 2, 11], max_selections=4, key='example_symbols_select_tab3') # Añadir emoji
example_symbols = constellation_mod[example_symbol_indices]


# Creamos una señal I y Q por partes, manteniendo constante el nivel de I y Q por símbolo
# Asegurarse de que haya al menos un símbolo seleccionado
if len(example_symbols) > 0:
    symbols_per_interval = len(example_symbols)
    samples_per_symbol_mod = int(len(t_qam_mod) / symbols_per_interval)
    # Asegurarse de que samples_per_symbol_mod sea al menos 1
    samples_per_symbol_mod = max(1, samples_per_symbol_mod)


    signal_i_mod = np.zeros_like(t_qam_mod)
    signal_q_mod = np.zeros_like(t_qam_mod)

    current_sample = 0
    for i in range(symbols_per_interval):
        start_sample = current_sample
        # Calcular el final de la muestra para este símbolo, asegurando no exceder el tamaño total
        end_sample = min(start_sample + samples_per_symbol_mod, len(t_qam_mod))

        if start_sample < end_sample: # Asegurar que hay al menos una muestra para este símbolo
             signal_i_mod[start_sample:end_sample] = example_symbols[i].real
             signal_q_mod[start_sample:end_sample] = example_symbols[i].imag
             current_sample = end_sample # Actualizar el inicio para el próximo símbolo
        else:
             # Si no hay suficientes muestras para asignar a samples_per_symbol_mod para el último símbolo,
             # simplemente no se asigna nada a ese símbolo, o se extiende el último símbolo válido.
             # En este caso simple, si end_sample no avanzó, detenemos el bucle.
             break


    # Modulación QAM: señal I * cos(2*pi*fc*t) - señal Q * sin(2*pi*fc*t)
    # La señal I modula la portadora coseno (En-fase)
    # La señal Q modula la portadora seno (En Cuadratura)
    qam_modulated_signal = signal_i_mod * np.cos(2 * np.pi * carrier_freq * t_qam_mod) - \
                           signal_q_mod * np.sin(2 * np.pi * carrier_freq * t_qam_mod)


    # Graficar Señales I(t), Q(t) para Modulación
    st.subheader("Señales en Fase (I) y Cuadratura (Q) para Modulación 📊") # Añadir emoji
    st.write("Estas son las componentes I(t) y Q(t) que modulan las portadoras coseno y seno. Cada nivel constante representa un símbolo QAM.")
    fig_iq_mod_time, ax_iq_mod_time = plt.subplots()
    ax_iq_mod_time.plot(t_qam_mod, signal_i_mod, label='I(t)')
    ax_iq_mod_time.plot(t_qam_mod, signal_q_mod, label='Q(t)')
    ax_iq_mod_time.set_xlabel("Tiempo [s]")
    ax_iq_mod_time.set_ylabel("Amplitud")
    ax_iq_mod_time.set_title("Señales I(t) y Q(t) para Modulación QAM (Ejemplo de Símbolos)")
    ax_iq_mod_time.legend()
    ax_iq_mod_time.grid(True)
    st.pyplot(fig_iq_mod_time)

    # Graficar Señal QAM Modulada en el Tiempo
    st.subheader("Señal QAM Modulada en el Tiempo 🌊") # Añadir emoji
    fig_qam_time, ax_qam_time = plt.subplots()
    ax_qam_time.plot(t_qam_mod, qam_modulated_signal)
    ax_qam_time.set_xlabel("Tiempo [s]")
    ax_qam_time.set_ylabel("Amplitud")
    ax_qam_time.set_title("Señal QAM Modulada (Ejemplo)")
    ax_qam_time.grid(True)
    st.pyplot(fig_qam_time)

    # Graficar Espectro de la Señal QAM Modulada
    yf_qam_mod = fft(qam_modulated_signal)
    xf_qam_mod = fftfreq(len(qam_modulated_signal), 1/fs_qam_mod)
    st.subheader("Espectro de la Señal QAM Modulada 📈") # Añadir emoji
    fig_qam_freq, ax_qam_freq = plt.subplots()
    # Graficamos el espectro completo, centrado en 0 Hz
    ax_qam_freq.plot(xf_qam_mod, 2.0/len(qam_modulated_signal) * np.abs(yf_qam_mod))
    ax_qam_freq.set_xlabel("Frecuencia [Hz]")
    ax_qam_freq.set_ylabel("Amplitud Normalizada")
    ax_qam_freq.set_title("Espectro de Frecuencia de la Señal QAM Modulada")
    ax_qam_freq.grid(True)
    # Opcional: limitar el rango de frecuencia mostrado
    ax_qam_freq.set_xlim([-carrier_freq*2, carrier_freq*2]) # Mostrar alrededor de la portadora
    st.pyplot(fig_qam_freq)
else:
    st.warning("Selecciona al menos un índice de símbolo de ejemplo para generar la señal. ⚠️") # Añadir emoji

Overwriting pages/3_Modulacion_QAM.py


**Reasoning**:
Create the fourth Python file in the 'pages' directory for the complete QAM system simulation.



In [ ]:
%%writefile pages/4_Sistema_Completo_QAM.py
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.spatial.distance import cdist # Importar para calcular distancias (para demodulador básico)

st.set_page_config(
    page_title="Sistema Completo QAM",
    page_icon="🔌", # Emoji para sistema completo
    layout="wide"
)

st.markdown("# 4. El Sistema Completo (QAM con Ruido y Demodulación) 🔌") # Añadir emoji al título principal
st.write(
    """
    Esta demostración ilustra el efecto del canal con Ruido Blanco Gaussiano Aditivo (AWGN)
    en una señal QAM y el resultado de un demodulador básico.
    Utiliza los controles en la barra lateral para ajustar
    el orden de la modulación QAM (M) y la relación Señal a Ruido (SNR).
    """
)

st.markdown("### Conceptos Clave en esta Sección ✨") # Añadir emoji
st.markdown("""
- **Canal de Comunicación:** El medio por el cual se transmite la señal. En comunicaciones inalámbricas, el aire es el canal, y está sujeto a diversas degradaciones. 📡💨
- **Ruido Blanco Gaussiano Aditivo (AWGN):** Un modelo simple pero fundamental de ruido que se añade a la señal. Es "blanco" porque tiene potencia uniforme en todas las frecuencias relevantes, y "Gaussiano" porque sus valores siguen una distribución normal. 👻📉
- **Demodulación:** El proceso inverso a la modulación, donde el receptor intenta recuperar la información original a partir de la señal recibida. En QAM, esto implica decidir qué punto de la constelación fue transmitido basándose en el símbolo recibido con ruido. 🧠➡️📍
- **Demodulador Básico (Nearest Neighbor):** Un demodulador simple que decide que el símbolo transmitido fue el punto de la constelación ideal más cercano al símbolo recibido en el plano I/Q. 📍📏
- **Diagrama de Constelación (Recibida):** Muestra los puntos de la constelación después de pasar por el canal con ruido. Los puntos se dispersan alrededor de sus posiciones ideales. ✨🌫️
- **Tasa de Error de Símbolo (SER):** La proporción de símbolos transmitidos que son demodulados incorrectamente. Es una medida clave del rendimiento del sistema. 📊❌
""")


# --- Controles para QAM del Sistema Completo (Mover aquí desde el sidebar o mantener allá) ---
# Decidimos mantenerlos en el sidebar para que sean accesibles sin importar la pestaña
st.sidebar.header("Parámetros del Sistema Completo") # Título de parámetros QAM en español (distinto del anterior)
qam_order_sys = st.sidebar.selectbox(
    "Selecciona Orden QAM (M) 🔢", # Añadir emoji
     [4, 16, 64, 256],
    index=1, # Default a 16-QAM
    key='sidebar_qam_order_sys' # Clave única para evitar conflictos
)
snr_db_sys = st.sidebar.slider(
    "Selecciona Relación Señal a Ruido (SNR en dB) 🔊", # Añadir emoji
    -10, 20, 10, # Min, Max, Default
    key='sidebar_snr_db_sys' # Clave única
)

# Número de símbolos a simular (Podría ser un slider también si se desea)
num_symbols_sys = st.slider("Número de Símbolos a Simular 📈", 1000, 50000, 10000, key='num_symbols_slider_tab4') # Añadir emoji y slider


# --- Función de Simulación QAM con Ruido ---
@st.cache_data # Cachear los resultados de la simulación para no recalcular si los parámetros no cambian
def simulate_qam_with_noise(qam_order, snr_db, num_symbols):
    """
    Simula transmisión QAM sobre un canal AWGN y retorna símbolos transmitidos,
    recibidos y constelación ideal.
    """
    # Generate random integer symbols
    input_bits_per_symbol = int(np.log2(qam_order))
    input_symbols = np.random.randint(0, qam_order, num_symbols)

    # Define the QAM constellation (simple square constellation mapping)
    # Normalize average energy to 1
    if qam_order == 4: # QPSK
       constellation = np.array([-1-1j, -1+1j, 1+1j, 1-1j]) * 1/np.sqrt(2)
    elif qam_order == 16:
       constellation = np.array([
         -3-3j, -3-1j, -3+3j, -3+1j,
         -1-3j, -1-1j, -1+3j, -1+1j,
          3-3j, 3-1j, 3+3j, 3+1j,
         1-3j, 1-1j, 1+3j, 1+1j
        ]) * 1/np.sqrt(10)
    elif qam_order == 64:
        pam_levels = np.arange(-(np.sqrt(qam_order)-1), np.sqrt(qam_order), 2)
        constellation = (pam_levels[:, np.newaxis] + 1j * pam_levels[np.newaxis, :]).flatten()
        constellation /= np.sqrt(((np.abs(constellation)**2).mean())) # Normalizar energía
    elif qam_order == 256:
        pam_levels = np.arange(-(np.sqrt(qam_order)-1), np.sqrt(qam_order), 2)
        constellation = (pam_levels[:, np.newaxis] + 1j * pam_levels[np.newaxis, :]).flatten()
        constellation /= np.sqrt(((np.abs(constellation)**2).mean())) # Normalizar energía
    else:
        st.error(f"Orden QAM {qam_order} no soportado en este ejemplo básico.")
        return None, None, None, None # Añadir None para SER si hay error

    # Map input symbols to constellation points
    transmitted_symbols = constellation[input_symbols]

    # Calculate noise power
    snr_linear = 10**(snr_db / 10.0)
    if snr_linear <= 1e-9: # Usar un umbral muy bajo para considerar SNR infinita (sin ruido efectivo)
        noise_power = 0.0 # No hay ruido
    else:
        # Signal power is normalized to 1, so noise power is 1/SNR_linear
        noise_power = 1 / snr_linear

    # Generate AWGN (complex noise)
    if noise_power == 0.0:
         noise = np.zeros(num_symbols, dtype=complex)
    else:
        noise_std_dev = np.sqrt(noise_power / 2) # Divide by 2 for complex noise (real and imag parts)
        noise = (noise_std_dev * np.random.randn(num_symbols)) + \
                (noise_std_dev * np.random.randn(num_symbols) * 1j)

    received_symbols = transmitted_symbols + noise

    return transmitted_symbols, received_symbols, constellation, input_symbols # También retornar input_symbols


    # --- Implementación de Demodulador Básico (Nearest Neighbor) ---
# No cacheamos el demodulador porque depende de los símbolos recibidos que vienen de la función cacheada
def basic_demodulator(received_symbols, ideal_constellation):
    """
    Demodulador básico que mapea cada símbolo recibido al punto de constelación
    ideal más cercano.
    Retorna los símbolos decididos (demodulados).
    """
    if received_symbols is None or len(received_symbols) == 0:
        return np.array([]) # Retorna un array vacío si no hay símbolos recibidos

    # Convertir a arrays de numpy para cdist
    received_points = np.array([[s.real, s.imag] for s in received_symbols])
    ideal_points = np.array([[c.real, c.imag] for c in ideal_constellation])

    if len(ideal_points) == 0:
         st.error("La constelación ideal está vacía.")
         return np.array([])

    # Calcular la distancia euclidiana de cada punto recibido a cada punto ideal de la constelación
    distances = cdist(received_points, ideal_points)

    # Encontrar el índice del punto ideal más cercano para cada punto recibido
    nearest_ideal_indices = np.argmin(distances, axis=1)

    # Obtener los símbolos decididos (los puntos ideales correspondientes a los índices encontrados)
    demodulated_symbols = ideal_constellation[nearest_ideal_indices]
    return demodulated_symbols, nearest_ideal_indices # Retornar también los índices decididos

# --- Calcular Tasa de Error de Símbolo (SER) ---
def calculate_ser(original_symbols_indices, demodulated_symbols_indices):
     """
     Calcula la Tasa de Error de Símbolo (SER).
     Compara los índices de los símbolos originales con los índices decididos por el demodulador.
     """
     if len(original_symbols_indices) == 0:
         return 0.0 # No hay símbolos para comparar

     # Contar cuántos símbolos demodulados no coinciden con los originales
     num_errors = np.sum(original_symbols_indices != demodulated_symbols_indices)

     # Calcular la SER
     ser = num_errors / len(original_symbols_indices)
     return ser


# --- Ejecutar Simulación del Sistema Completo y Mostrar Resultados ---
st.subheader("Resultados de la Simulación")

# Use st.spinner for simulation time
with st.spinner(f'Simulando sistema completo con {num_symbols_sys} símbolos para {qam_order_sys}-QAM con {snr_db_sys} dB SNR...'):
    transmitted_symbols_sys, received_symbols_sys, ideal_constellation_sys, original_symbols_indices_sys = simulate_qam_with_noise(qam_order_sys, snr_db_sys, num_symbols_sys)
    if received_symbols_sys is not None and ideal_constellation_sys is not None and original_symbols_indices_sys is not None:
         demodulated_symbols_sys, demodulated_symbols_indices_sys = basic_demodulator(received_symbols_sys, ideal_constellation_sys)

         # Calcular SER solo si la demodulación fue exitosa y hay símbolos originales
         if len(original_symbols_indices_sys) > 0 and len(demodulated_symbols_indices_sys) == len(original_symbols_indices_sys):
             ser_sys = calculate_ser(original_symbols_indices_sys, demodulated_symbols_indices_sys)
             st.write(f"Simulación del sistema completo realizada con {num_symbols_sys} símbolos, {qam_order_sys}-QAM y SNR = {snr_db_sys} dB.")
             st.metric("Tasa de Error de Símbolo (SER) 📉", f"{ser_sys:.6f}") # Mostrar la SER con emoji
         else:
             st.warning("No se pudo calcular la SER debido a un problema en la simulación o demodulación.")
             st.write(f"Simulación del sistema completo realizada con {num_symbols_sys} símbolos, {qam_order_sys}-QAM y SNR = {snr_db_sys} dB.")

    else:
         demodulated_symbols_sys = None
         demodulated_symbols_indices_sys = None
         st.error("La simulación no pudo completarse. Verifica los parámetros.")


# --- Graficar Diagrama de Constelación (Ideal, Recibida y Demodulada) ---
st.subheader("Diagrama de Constelación 🌌") # Añadir emoji
st.write("Este gráfico muestra los puntos de la constelación ideal, los símbolos recibidos afectados por el ruido, y los símbolos decididos por el demodulador básico.")

if received_symbols_sys is not None and demodulated_symbols_sys is not None and ideal_constellation_sys is not None:
    fig_constellation_sys, ax_constellation_sys = plt.subplots(figsize=(8, 8))

    # Mostrar la constelación ideal
    ax_constellation_sys.scatter(ideal_constellation_sys.real, ideal_constellation_sys.imag,
                                 s=150, color='red', marker='*', edgecolors='black',
                                 label='Símbolos Ideales', zorder=3) # zorder para que se vean encima

    # Mostrar la constelación recibida con ruido
    ax_constellation_sys.scatter(received_symbols_sys.real, received_symbols_sys.imag,
                                 s=20, alpha=0.5, label=f"Símbolos Recibidos con Ruido (SNR={snr_db_sys} dB)", zorder=1) # s reducido, alpha para densidad

    # Mostrar la constelada (decisiones del demodulador)
    ax_constellation_sys.scatter(demodulated_symbols_sys.real, demodulated_symbols_sys.imag,
                                 s=100, color='green', marker='x',
                                 label="Símbolos Demodulados (Decisiones)", zorder=2) # zorder intermedio

    ax_constellation_sys.set_title(f"Constelación {qam_order_sys}-QAM")
    ax_constellation_sys.set_xlabel("En-fase (I)")
    ax_constellation_sys.set_ylabel("Cuadratura (Q)")
    ax_constellation_sys.grid(True)
    ax_constellation_sys.set_aspect('equal', adjustable='box') # Escalado uniforme

    # Ajustar límites automáticamente basado en los puntos ideal y recibido
    all_points_sys = np.concatenate((ideal_constellation_sys, received_symbols_sys))
    if len(all_points_sys) > 0 and np.isfinite(all_points_sys).all():
         max_val_sys = np.max(np.abs(all_points_sys)) * 1.2
         ax_constellation_sys.set_xlim(-max_val_sys, max_val_sys)
         ax_constellation_sys.set_ylim(-max_val_sys, max_val_sys)
    else:
         # Fallback si los puntos son infinitos o no hay puntos
         max_val_ideal = np.max(np.abs(ideal_constellation_sys)) * 1.5 if len(ideal_constellation_sys) > 0 else 1
         ax_constellation_sys.set_xlim(-max_val_ideal, max_val_ideal)
         ax_constellation_sys.set_ylim(-max_val_ideal, max_val_ideal)


    ax_constellation_sys.legend(loc='upper right')
    fig_constellation_sys.tight_layout()
    st.pyplot(fig_constellation_sys)

    # Mostrar tabla con primeros 10 símbolos
    st.subheader("Datos de Ejemplo del Sistema (Primeros 10 Símbolos) 📋") # Añadir emoji
    sample_df_sys = pd.DataFrame({
        'Ideal (I)': ideal_constellation_sys[:10].real,
        'Ideal (Q)': ideal_constellation_sys[:10].imag,
        'Recibido (I)': received_symbols_sys[:10].real,
        'Recibido (Q)': received_symbols_sys[:10].imag,
        'Demodulado (I)': demodulated_symbols_sys[:10].real,
        'Demodulado (Q)': demodulated_symbols_sys[:10].imag,
    })
    st.dataframe(sample_df_sys)

elif ideal_constellation_sys is not None:
     # Si solo se pudo generar la constelación ideal (ej. SNR muy baja resultó en Inf)
     fig_constellation_sys, ax_constellation_sys = plt.subplots(figsize=(8, 8))
     ax_constellation_sys.scatter(ideal_constellation_sys.real, ideal_constellation_sys.imag,
                                 s=150, color='red', marker='*', edgecolors='black',
                                 label='Símbolos Ideales', zorder=3)
     ax_constellation_sys.set_title(f"Constelación {qam_order_sys}-QAM (Solo Ideal)")
     ax_constellation_sys.set_xlabel("En-fase (I)")
     ax_constellation_sys.set_ylabel("Cuadratura (Q)")
     ax_constellation_sys.grid(True)
     ax_constellation_sys.set_aspect('equal', adjustable='box')
     max_val_ideal = np.max(np.abs(ideal_constellation_sys)) * 1.5 if len(ideal_constellation_sys) > 0 else 1
     ax_constellation_sys.set_xlim(-max_val_ideal, max_val_ideal)
     ax_constellation_sys.set_ylim(-max_val_ideal, max_val_ideal)
     ax_constellation_sys.legend(loc='upper right')
     fig_constellation_sys.tight_layout()
     st.pyplot(fig_constellation_sys)
else:
     st.warning("No hay datos de simulación para mostrar la constelación.")


# --- Nueva Sección: Aplicación en WiFi y 5G ---
st.markdown("## Aplicación de QAM en WiFi y 5G 🌐📱") # Título de la nueva sección con emojis
st.write(
    """
    La Modulación de Amplitud en Cuadratura (QAM) es fundamental en estándares modernos de comunicaciones
    inalámbricas como **WiFi (IEEE 802.11)** y **5G**. Permite transmitir múltiples bits por cada símbolo
    transmitido, aumentando significativamente la tasa de datos que se puede enviar a través del aire.

    En sistemas como WiFi y 5G, QAM se utiliza en combinación con otras técnicas como **OFDM (Orthogonal Frequency-Division Multiplexing)**.
    OFDM divide el canal de comunicación en muchas subportadoras estrechas, y cada una de estas subportadoras
    puede ser modulada de forma independiente utilizando esquemas como QAM. Esto ayuda a combatir los efectos
    de la dispersión multitrayecto y mejorar la eficiencia espectral.

    **¿Cómo se integra QAM en un sistema real (ej. WiFi)?**

    Piensa en el diagrama de bloques general de un transmisor/receptor.

    **En el Transmisor:**
    1.  Los datos binarios se agrupan en bloques (por ejemplo, 4 bits para 16-QAM).
    2.  Cada bloque de bits se mapea a un punto específico en la constelación QAM (esto define los valores de I y Q).

    3.  Estos valores de I y Q modulan las portadoras coseno y seno desfasadas 90 grados.
    4.  La señal modulada se combina con otras señales (si se usa OFDM) y se transmite.

    """
)

# Código para mostrar la imagen del transmisor
# Modificar el enlace de Google Drive para descarga directa
transmitter_drive_id = "1-TjbaWXZWvO5FpH6mu6UN4YOIt07C9UY"
transmitter_url = f"https://drive.google.com/uc?export=download&id={transmitter_drive_id}"
try:
    st.image(transmitter_url, caption="Diagrama de Bloques Transmisor Simplificado", use_container_width=True)
except Exception as e:
    st.warning(f"No se pudo cargar la imagen del transmisor desde Google Drive. Asegúrate de que el enlace sea público. Error: {e}")


st.markdown(
    """
    **En el Receptor:**
    1.  La señal recibida (que incluye ruido y otras interferencias) se demodula. Esto implica separar las componentes I y Q.
    2.  El demodulador (como el que simulamos aquí usando la proximidad en el plano I/Q) intenta decidir qué punto de la constelación fue el más probable que se transmitió.

    3.  El punto de constelación decidido se mapea de vuelta a la secuencia de bits original.
    4.  Estos bits recuperados son los datos recibidos.

    """
)

# Código para mostrar la imagen del receptor
# Modificar el enlace de Google Drive para descarga directa
receiver_drive_id = "1m56QQPcueH25GMXoGRxAIhaVPzvsPlLG"
receiver_url = f"https://drive.google.com/uc?export=download&id={receiver_drive_id}"
try:
    st.image(receiver_url, caption="Diagrama de Bloques Receptor Simplificado", use_container_width=True)
except Exception as e:
    st.warning(f"No se pudo cargar la imagen del receptor desde Google Drive. Asegúrate de que el enlace sea público. Error: {e}")


st.markdown(
    """
    La elección del orden QAM (4-QAM, 16-QAM, 64-QAM, 256-QAM, etc.) depende de las condiciones del canal (principalmente la SNR). Con una SNR alta, se pueden usar órdenes QAM más altas para transmitir más datos por símbolo (mayor velocidad), pero son más susceptibles al ruido (mayor SER). Con una SNR baja, se usan órdenes QAM más bajas (como 4-QAM o QPSK) que son más robustas al ruido pero transmiten menos datos.

    **Visualizaciones Adicionales para el Video:**
    Para tu video, podrías considerar incluir:
    *   Un diagrama de bloques simplificado del transmisor y receptor que muestre dónde encaja la modulación/demodulación QAM (ya añadimos el del transmisor y receptor si los archivos están disponibles).
    *   Un GIF o animación que muestre cómo los puntos de la constelación se dispersan con diferentes niveles de ruido (cambiando el slider de SNR en la simulación - ¡la constelación interactiva ya te permite mostrar esto!).
    *   Una tabla o gráfico que muestre cómo la SER cambia a medida que varías la SNR (la métrica SER ya está visible y cambia con el slider de SNR - ¡puedes narrar esto!).
    *   Imágenes o diagramas que ilustren el concepto de OFDM y cómo QAM se aplica a cada subportadora.

    ¡Utiliza esta sección y la interactividad del dashboard como base para narrar tu explicación en el video!
    """
)

# Puedes añadir aquí placeholders para imágenes si tienes URLs o rutas locales
# st.image("ruta/a/tu/gif_constelacion_snr.gif", caption="Efecto del Ruido en la Constelación QAM") # Ya es interactivo
# st.image("ruta/a/tu/diagrama_ofdm.png", caption="Concepto de OFDM con QAM")

Overwriting pages/4_Sistema_Completo_QAM.py


In [ ]:
%%writefile 0_👋_Hello.py
import streamlit as st
st.set_page_config(
    page_title="Dashboard Proyecto Comunicaciones Digitales",
    page_icon="👋",
)

st.write("# Dashboard del Proyecto de Comunicaciones Digitales 👋")

# Update sidebar success message to refer to selecting pages from the sidebar
st.sidebar.success("Selecciona una página de simulación de la barra lateral para explorar las diferentes fases del proyecto. 👉")

st.markdown(
    """
    ## Introducción ✨

    ¡Hola y bienvenido a nuestro dashboard interactivo para el proyecto final del curso de **Señales y Sistemas**! 👋

    Este proyecto, titulado **"De Fourier al WiFi/5G: Anatomía de una Señal Inalámbrica"** 📡, explora cómo los principios fundamentales de la teoría de señales y sistemas son la base de las comunicaciones inalámbricas modernas. A través de simulaciones y visualizaciones 📊, buscamos desmitificar la "magia" detrás de tecnologías como Wi-Fi y 5G.

    ## Contenido del Dashboard 🔬

    El dashboard está dividido en secciones clave que te permitirán interactuar con diferentes simulaciones y conceptos, **disponibles como páginas separadas en la barra lateral**:

    *   **1. El Dominio de la Frecuencia:** Explora la Transformada Rápida de Fourier (FFT) y el diseño/aplicación de filtros digitales. 📉
    *   **2. Construyendo las Señales I/Q:** Comprende cómo obtener las componentes en fase (I) y cuadratura (Q) utilizando la Transformada de Hilbert. 📈
    *   **3. Modulación QAM:** Visualiza el proceso de modulación QAM, sus señales I/Q correspondientes, su espectro y el diagrama de constelación. 🌌
    *   **4. El Sistema Completo:** Simula el efecto del ruido en la señal QAM y observa cómo un demodulador básico intenta recuperar los símbolos, visualizando la constelación recibida y demodulada. 🔌

    ## Información del Proyecto 🎓

    Este trabajo fue desarrollado en la **Universidad Nacional de Colombia - sede Manizales** 🏛️ como proyecto final de la asignatura **Señales y Sistemas**.

    *   **Profesor:** Andrés Marino Álvarez Meza, Ph.D. 👨‍🏫
    *   **Integrantes del Equipo:**
     Daniel Santiengo Escruceria Rozero 🧑‍💻
     Alexis Valencia 🧑‍💻
     Darwin Andrey Arias Garcia 🧑‍💻

    ---

    **¡Te invitamos a explorar!** 👇

    **👈 Selecciona una de las páginas de simulación en la barra lateral** para sumergirte en las simulaciones y ver la teoría de señales y sistemas en acción aplicada a las comunicaciones digitales.
    """
)

Overwriting 0_👋_Hello.py


In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!mv cloudflared-linux-amd64 /usr/local/bin/cloudflared
#Ejecutar Streamlit
!streamlit run 0_👋_Hello.py &>/content/logs.txt & #Cambiar 0_👋_Hello.py por el nombre de tu archivo principal
#Exponer el puerto 8501 con Cloudflare Tunnel
!cloudflared tunnel --url http://localhost:8501 > /content/cloudflared.log 2>&1 &
#Leer la URL pública generada por Cloudflare
import time
time.sleep(10) # Esperar que se genere la URL
import re
found_context = False # Indicador para saber si estamos en la sección correcta
with open('/content/cloudflared.log') as f:
    for line in f:
        #Detecta el inicio del contexto que nos interesa
        if "Your quick Tunnel has been created" in line:
            found_context = True
        #Busca una URL si ya se encontró el contexto relevante
        if found_context:
            match = re.search(r'https?://\S+', line)
            if match:
                url = match.group(0) #Extrae la URL encontrada
                print(f'Tu aplicación está disponible en: {url}')
                break #Termina el bucle después de encontrar la URL

--2025-07-25 02:05:32--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2025.7.0/cloudflared-linux-amd64 [following]
--2025-07-25 02:05:32--  https://github.com/cloudflare/cloudflared/releases/download/2025.7.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/37d2bad8-a2ed-4b93-8139-cbb15162d81d?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-07-25T02%3A54%3A37Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-07-25T0

In [ ]:
import os
res = input("Digite (1) para finalizar la ejecución del Dashboard: ")
if res.upper() == "1":
    os.system("pkill streamlit") # Termina el proceso de Streamlit
    print("El proceso de Streamlit ha sido finalizado.")